# TFT Training — Lim et al. 2021 Replica

Paper Electricity hyperparams (Table 6): hidden 160, dropout 0.1, attention heads 4, lr 0.001, batch 64, grad clip 0.01, encoder 168h, quantiles (0.1, 0.5, 0.9), QuantileLoss.

**One deviation:** decoder = 120h (client 1–5 day UI). Paper used 24h. Everything else paper-exact.

Run from `Shahriar/`. Needs `tft_hourly.parquet`.

In [1]:
# %pip install pytorch-forecasting lightning torch mlflow python-dotenv pyarrow

In [2]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from dotenv import load_dotenv
load_dotenv(".env")
pl.seed_everything(42)
print("cuda:", torch.cuda.is_available())

Seed set to 42


cuda: True


In [3]:
df = pd.read_parquet("tft_hourly.parquet")
df = df[df.ts >= "2021-03-23"].reset_index(drop=True)
df["time_idx"] = df["time_idx"] - df["time_idx"].min()
for c in ["hour","day_of_week","day_of_month","month","is_holiday","weather_is_forecast"]:
    df[c] = df[c].astype(str).astype("category")
df["series"] = df["series"].astype("category")
train_df = df[df.split=="train"]; test_df = df[df.split=="test"]
print(train_df.shape, test_df.shape)

(24337, 100) (21120, 100)


In [4]:
ENC, DEC = 168, 120
LEADS = [1,2,3,4,5]
BASEVARS = ["temperature_2m","relative_humidity_2m","dew_point_2m","apparent_temperature",
            "precipitation","rain","snowfall","cloud_cover","surface_pressure",
            "wind_speed_10m","wind_gusts_10m","shortwave_radiation",
            "direct_radiation","diffuse_radiation"]
WEATHER = BASEVARS + [f"{v}_previous_day{k}" for v in BASEVARS for k in LEADS]
WEATHER = [c for c in WEATHER if c in df.columns]
print("weather cols:", len(WEATHER))
GEN = [c for c in ["net_generation","total_interchange","ng_nuclear","ng_hydro",
                   "ng_solar","ng_wind","ng_natural_gas"] if c in df.columns]

val_cut = train_df.time_idx.max() - 24*90
training = TimeSeriesDataSet(
    train_df[train_df.time_idx <= val_cut],
    time_idx="time_idx",
    target="demand",
    group_ids=["series"],
    max_encoder_length=ENC,
    max_prediction_length=DEC,
    static_categoricals=["series"],
    time_varying_known_categoricals=["hour","day_of_week","day_of_month","month","is_holiday"],
    time_varying_known_reals=["time_idx"] + WEATHER,
    time_varying_unknown_reals=["demand"] + GEN + ["NYNGSP","NYPOP"],
    target_normalizer=GroupNormalizer(groups=["series"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    allow_missing_timesteps=False,
)
validation = TimeSeriesDataSet.from_dataset(training, train_df, predict=False,
              stop_randomization=True, min_prediction_idx=val_cut+1)
BATCH = 64
train_dl = training.to_dataloader(train=True, batch_size=BATCH, num_workers=0)
val_dl = validation.to_dataloader(train=False, batch_size=BATCH*4, num_workers=0)

weather cols: 84


In [5]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    hidden_size=160,
    lstm_layers=1,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=160,
    learning_rate=0.001,
    loss=QuantileLoss([0.1, 0.5, 0.9]),
    log_interval=50,
    reduce_on_plateau_patience=3,
)
print(f"params: {tft.size()/1e6:.2f}M")

params: 22.56M


In [6]:
ckpt = ModelCheckpoint(monitor="val_loss", mode="min", dirpath="../models",
                       filename="tft_best")
trainer = pl.Trainer(
    max_epochs=100,
    accelerator="gpu", devices=1,
    gradient_clip_val=0.01,
    callbacks=[EarlyStopping(monitor="val_loss", patience=10), ckpt],
    enable_progress_bar=True,
)
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)

Epoch 13/99 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342/342 0:04:58 • 0:00:00 1.15it/s v_num: 12.000 train_loss_step:    
                                                                                 63.315 val_loss: 249.894          
                                                                                 train_loss_epoch: 57.593          

In [7]:
best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)

test_ds = TimeSeriesDataSet.from_dataset(training, df, predict=False, stop_randomization=True,
                                         min_prediction_idx=train_df.time_idx.max()+1)
test_dl = test_ds.to_dataloader(train=False, batch_size=256, num_workers=0)
raw = best.predict(test_dl, mode="quantiles", return_x=True)
p50 = raw.output[..., 1].cpu().numpy()
y = raw.x["decoder_target"].cpu().numpy()

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [8]:
def split_metrics(dl, label):
    r = best.predict(dl, mode="quantiles", return_x=True)
    p = r.output[..., 1].cpu().numpy()
    yy = r.x["decoder_target"].cpu().numpy()
    rows=[]
    for d in range(5):
        sl = slice(d*24, (d+1)*24)
        a, b = yy[:, sl].ravel(), p[:, sl].ravel()
        rows.append({"split": label, "day": d+1,
                     "RMSE": float(np.sqrt(np.mean((a-b)**2))),
                     "MAE": float(np.abs(a-b).mean()),
                     "MAPE_%": float(np.mean(np.abs((a-b)/a))*100)})
    return pd.DataFrame(rows), r

train_eval_dl = training.to_dataloader(train=False, batch_size=256, num_workers=0)
m_tr, _   = split_metrics(train_eval_dl, "train")
m_va, _   = split_metrics(val_dl, "val")
m_te, raw = split_metrics(test_dl, "test")
summary = pd.concat([m_tr, m_va, m_te]).set_index(["split","day"])
print(summary.round(2))

p50 = raw.output[...,1].cpu().numpy()
p10, p90 = raw.output[...,0].cpu().numpy(), raw.output[...,2].cpu().numpy()
y = raw.x["decoder_target"].cpu().numpy()
def q_risk(q, pred):
    e = y - pred
    return 2*np.sum(np.maximum(q*e, (q-1)*e)) / np.sum(np.abs(y))
print(f"P50 q-risk: {q_risk(0.5,p50):.4f}  P90 q-risk: {q_risk(0.9,p90):.4f}")

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


              RMSE     MAE  MAPE_%
split day                         
train 1     314.98  236.73    1.34
      2     271.75  204.79    1.15
      3     266.36  199.03    1.11
      4     269.41  201.20    1.13
      5     284.06  211.26    1.18
val   1     429.20  312.39    1.93
      2     434.80  313.95    1.94
      3     445.21  321.76    1.99
      4     452.25  329.32    2.03
      5     461.52  339.48    2.10
test  1     913.79  682.65    3.84
      2     980.83  726.81    4.07
      3     996.57  736.09    4.12
      4    1006.93  741.82    4.14
      5    1015.39  745.60    4.16
P50 q-risk: 0.0423  P90 q-risk: 0.0406


In [9]:
import mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("DAGSHUB_USER_NAME","")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("DAGSHUB_TOKEN","")
uri = os.getenv("MLFLOW_TRACKING_URI")
if uri:
    mlflow.set_tracking_uri(uri); mlflow.set_experiment("tft")
    with mlflow.start_run(run_name="tft_paper_replica"):
        mlflow.log_params({"model":"TFT","hidden":160,"heads":4,"dropout":0.1,
                           "lr":0.001,"batch":64,"grad_clip":0.01,
                           "encoder_h":168,"decoder_h":120,"quantiles":"0.1/0.5/0.9",
                           "paper":"arXiv:1912.09363","weather":"stitched+perlead1-5"})
        mlflow.log_metric("p50_qrisk", float(q_risk(0.5,p50)))
        mlflow.log_metric("p90_qrisk", float(q_risk(0.9,p90)))
        for (s, d) in summary.index:
            for m in ["RMSE","MAE","MAPE_%"]:
                mlflow.log_metric(f"{s}_{m}_day{d}".replace("%","pct"), float(summary.loc[(s,d), m]))
        mlflow.log_artifact(ckpt.best_model_path)
    print("logged")
else:
    print("no MLFLOW_TRACKING_URI")

🏃 View run tft_paper_replica at: https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow/#/experiments/3/runs/1205b01deaca4980b1f0b45e6b1d3fb3
🧪 View experiment at: https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow/#/experiments/3
logged


In [ ]:
import torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.cuda.empty_cache()

best = TemporalFusionTransformer.load_from_checkpoint("../models/tft_best-v3.ckpt")

interp_dl = validation.to_dataloader(train=False, batch_size=32, num_workers=0)

raw_out = best.predict(interp_dl, mode="raw", return_x=True)
interp = best.interpret_output(raw_out.output, reduction="sum")

fig = best.plot_interpretation(interp)
import os
os.makedirs("../reports", exist_ok=True)
for name, f in fig.items():
    f.savefig(f"../reports/tft_importance_{name}.png", dpi=120, bbox_inches="tight")
    plt.close(f)
print("saved:", list(fig.keys()))